# UMLForge — benchmark de Ollama con GPU

Este cuaderno mide la velocidad de `qwen3.5:4b` en una GPU de Kaggle. Las últimas celdas permiten, opcionalmente, conectar UMLForge mediante un túnel temporal protegido con token.

Antes de ejecutarlo, activa **Settings → Accelerator → GPU** e **Internet → On**.

In [ ]:
# Comprueba que Kaggle asignó una GPU NVIDIA a esta sesión.
!nvidia-smi
!free -h

In [ ]:
# Instala la utilidad de descompresión que necesita el instalador actual de Ollama.
# Kaggle ejecuta el notebook como root, por eso no hace falta sudo.
!apt-get update -qq && apt-get install -y -qq zstd

# Instala Ollama en el contenedor efímero de Kaggle.
# Internet debe estar activado para esta celda.
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Inicia Ollama solo en localhost. No se expone a Internet.
import os
import subprocess
import time
import urllib.request

env = os.environ.copy()
env['OLLAMA_HOST'] = '127.0.0.1:11434'
ollama_process = subprocess.Popen(
    ['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama listo en localhost.')
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('Ollama no inició. Verifica la instalación y vuelve a iniciar la sesión.')

In [ ]:
# Descarga el mismo modelo configurado actualmente en UMLForge (~3.3 GB).
# Esta operación puede tardar varios minutos la primera vez.
MODEL = 'qwen3.5:4b'
result = subprocess.run(['ollama', 'pull', MODEL], text=True, capture_output=True)
print(result.stdout[-2000:])
if result.returncode:
    print(result.stderr[-2000:])
    raise RuntimeError('No se pudo descargar el modelo.')
!ollama list

In [ ]:
# Calentamiento: carga el modelo y verifica que Ollama ve la GPU.
import json
import requests

warmup = requests.post(
    'http://127.0.0.1:11434/api/generate',
    json={'model': MODEL, 'prompt': 'Responde solamente: listo', 'stream': False, 'options': {'num_predict': 8}},
    timeout=300,
).json()
print('Respuesta:', warmup.get('response'))
print(requests.get('http://127.0.0.1:11434/api/ps', timeout=10).json())
print('Busca size_vram mayor que 0: indica que el modelo está en GPU.')

In [ ]:
# Prueba representativa: una respuesta UML JSON corta, sin razonamiento.
# Se limita la salida a 256 tokens, comparable con una configuración optimizada para UMLForge.
prompt = '''Eres un asistente UML. Devuelve solo JSON válido con estas claves:
reply_to_user, canvas_patches.
Crea la clase Cliente con atributos id:int y email:String, y la clase Pedido con id:int.
canvas_patches debe contener operaciones ADD_CLASS.'''

started = time.perf_counter()
response = requests.post(
    'http://127.0.0.1:11434/api/generate',
    json={
        'model': MODEL,
        'prompt': prompt,
        'stream': False,
        'think': False,
        'format': 'json',
        'options': {'temperature': 0.2, 'num_predict': 256},
    },
    timeout=300,
).json()
wall_seconds = time.perf_counter() - started

def seconds(ns):
    return ns / 1_000_000_000

prompt_seconds = seconds(response.get('prompt_eval_duration', 0))
generation_seconds = seconds(response.get('eval_duration', 0))
prompt_tokens = response.get('prompt_eval_count', 0)
generated_tokens = response.get('eval_count', 0)
print(response.get('response', ''))
print('\n--- Resultado ---')
print(f'Latencia total: {wall_seconds:.2f} s')
print(f'Prompt: {prompt_tokens} tokens en {prompt_seconds:.2f} s ({prompt_tokens / prompt_seconds if prompt_seconds else 0:.2f} tok/s)')
print(f'Generación: {generated_tokens} tokens en {generation_seconds:.2f} s ({generated_tokens / generation_seconds if generation_seconds else 0:.2f} tok/s)')
print('GPU durante la prueba:')
subprocess.run(['nvidia-smi'], check=False)

## Cómo interpretar el resultado

- `size_vram > 0` en `/api/ps` confirma que Ollama cargó el modelo en GPU.
- Compara **Generación tok/s** con tu equipo local: allí se midieron aproximadamente **7.5 tok/s**.
- Ejecuta la última celda dos veces y usa la segunda: evita incluir la carga inicial del modelo.
- Al terminar, usa **Stop Session** en Kaggle para no gastar cuota GPU. Cada sesión tiene máximo 12 horas y la cuota semanal suele ser de hasta 30 horas.

## Opcional: conexión temporal con UMLForge

Ejecuta las dos celdas siguientes solo durante la demostración autorizada. Crean un proxy local protegido por un token y un túnel HTTPS temporal. La URL deja de servir cuando termina la sesión de Kaggle.

In [ ]:
# Dependencias del proxy (normalmente ya vienen en Kaggle).
!pip install -q fastapi uvicorn

# Proxy mínimo: solo permite POST /api/chat si recibe el token correcto.
# El token no se imprime ni se guarda en el notebook.
import getpass
import hmac
import socket
import threading
import time
import requests
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import Response
import uvicorn

try:
    requests.get('http://127.0.0.1:11434/api/tags', timeout=5).raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        'Ollama no está activo. Ejecuta primero la celda que inicia Ollama y espera el mensaje de listo.'
    ) from exc

REMOTE_TOKEN = getpass.getpass('Crea y escribe un token largo (mínimo 32 caracteres): ')
if len(REMOTE_TOKEN) < 32:
    raise ValueError('Usa un token de al menos 32 caracteres.')

proxy = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)

@proxy.post('/api/chat')
async def protected_chat(request: Request, authorization: str = Header(default='')):
    expected = f'Bearer {REMOTE_TOKEN}'
    if not hmac.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='Token inválido')
    upstream = requests.post(
        'http://127.0.0.1:11434/api/chat',
        data=await request.body(),
        headers={'Content-Type': 'application/json'},
        timeout=300,
    )
    return Response(content=upstream.content, status_code=upstream.status_code, media_type='application/json')

# Pide un puerto libre para que sea seguro volver a ejecutar la celda.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.bind(('127.0.0.1', 0))
    PROXY_PORT = sock.getsockname()[1]

proxy_server = uvicorn.Server(uvicorn.Config(proxy, host='127.0.0.1', port=PROXY_PORT, log_level='warning'))
threading.Thread(target=proxy_server.run, daemon=True).start()
time.sleep(2)
print(f'Proxy protegido listo solo en localhost:{PROXY_PORT}.')

In [ ]:
# Crea un túnel HTTPS temporal hacia el proxy local y muestra su URL.
# No compartas esta URL ni el token fuera de tu equipo/equipo de examen.
import re
import subprocess

subprocess.run([
    'wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PROXY_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
tunnel_url = None
for _ in range(90):
    line = tunnel_process.stdout.readline()
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
if not tunnel_url:
    raise RuntimeError('No se obtuvo la URL del túnel. Reinicia esta celda.')
# Seguir leyendo la salida evita que el túnel se bloquee al llenarse el pipe.
def drain_tunnel_output():
    for _ in tunnel_process.stdout:
        pass
threading.Thread(target=drain_tunnel_output, daemon=True).start()
print('\nURL temporal para backend/.env:')
print(tunnel_url)
print('\nMantén activa la sesión de Kaggle durante la demostración.')